<a href="https://colab.research.google.com/github/alee52/LLM_AgenticAI/blob/main/Fine_tuning_Llama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q --upgrade bitsandbytes
!pip install -q --upgrade trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.0 MB/s eta 0:00:00


In [2]:
import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig
from datasets import load_dataset, Dataset, DatasetDict
import wandb
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datetime import datetime
import matplotlib.pyplot as plt

In [3]:
import warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger("trl").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)

In [4]:
# Constants

BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "categorize_products"
HF_USER = "leearum95"

LITE_MODE = True

DATA_USER = "leearum95"
DATASET_NAME = f"{DATA_USER}/items_prompts_full"

RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
if LITE_MODE:
  RUN_NAME += "-lite"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# Hyper-parameters - overall

EPOCHS = 1 if LITE_MODE else 3
BATCH_SIZE = 32 if LITE_MODE else 256
MAX_SEQUENCE_LENGTH = 128
GRADIENT_ACCUMULATION_STEPS = 1

# Hyper-parameters - QLoRA

QUANT_4_BIT = True
LORA_R = 32 if LITE_MODE else 256
LORA_ALPHA = LORA_R * 2
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
MLP_LAYERS = ["gate_proj", "up_proj", "down_proj"]
TARGET_MODULES = ATTENTION_LAYERS if LITE_MODE else ATTENTION_LAYERS + MLP_LAYERS
LORA_DROPOUT = 0.1

# Hyper-parameters - training

LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = 'cosine'
WEIGHT_DECAY = 0.001
OPTIMIZER = "paged_adamw_32bit"

capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

# Tracking

VAL_SIZE = 500 if LITE_MODE else 1000
LOG_STEPS = 5 if LITE_MODE else 10
SAVE_STEPS = 100 if LITE_MODE else 200
LOG_TO_WANDB = True

In [5]:
use_bf16

False

In [6]:
# Log in to HuggingFace

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
# Log in to Weights & Biases
wandb_api_key = userdata.get('WANDB_API_KEY')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

# Configure Weights & Biases to record against our project
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

In [7]:
dataset = load_dataset(DATASET_NAME)
train = dataset['train']
val = dataset['val'].select(range(VAL_SIZE))
test = dataset['test']

README.md:   0%|          | 0.00/513 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.75M [00:00<?, ?B/s]

data/val-00000-of-00001.parquet:   0%|          | 0.00/753k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/587k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/4000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3099 [00:00<?, ? examples/s]

In [ ]:
if LOG_TO_WANDB:
  wandb.init(project=PROJECT_NAME, name=RUN_NAME)

# Load the Tokenizer and Model

In [8]:
if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
  )

In [9]:
# Load the Tokenizer and the Model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Memory footprint: 2197.6 MB


In [15]:
tokens = tokenizer.encode(train[0]["prompt"])
print(tokens)
print("\n")
print(tokenizer.decode(tokens))

[128000, 3923, 374, 279, 5699, 315, 279, 2768, 2027, 704, 315, 279, 2768, 11306, 25, 7972, 11, 17979, 612, 23334, 11, 56120, 612, 27399, 11, 8184, 612, 25563, 11, 53176, 612, 28764, 7439, 532, 11, 30708, 645, 11, 38784, 11, 22837, 27864, 11, 5492, 612, 19558, 11, 21266, 612, 30534, 1565, 11, 6401, 612, 31685, 11, 8471, 11, 65403, 42695, 11, 12369, 11, 39099, 1154, 612, 66210, 11, 37865, 11, 82700, 612, 40819, 11, 49481, 612, 11871, 11, 47966, 612, 11586, 51352, 11, 5761, 2758, 67454, 82, 11, 445, 44765, 612, 64823, 11, 89870, 612, 16963, 1233, 11, 82028, 11, 8410, 51352, 11, 4476, 11, 30339, 271, 14235, 4870, 323, 27220, 220, 1682, 25224, 305, 2402, 579, 29551, 88949, 16850, 11113, 36591, 23529, 11, 264, 9966, 5944, 5409, 11, 33236, 5419, 7471, 11, 323, 19307, 47056, 6812, 12811, 11, 10728, 369, 10273, 51597, 389, 25425, 11, 23277, 11, 477, 2626, 5944, 382, 791, 5699, 374, 220]


<|begin_of_text|>What is the category of the following product out of the following categories: Media, Arts

In [17]:
tokens = tokenizer.encode(train[0]["completion"])
print(tokens)
print("\n")
print(tokenizer.decode(tokens))

[128000, 43, 44765, 612, 64823]


<|begin_of_text|>Luggage & Bags


In [18]:
print(train[0])

{'prompt': 'What is the category of the following product out of the following categories: Media, Arts & Entertainment, Vehicles & Parts, Business & Industrial, Religious & Ceremonial, Bundles, Electronics, Gift Cards, Home & Garden, Baby & Toddler, Health & Beauty, Services, Sporting Goods, Food, Beverages & Tobacco, Hardware, Apparel & Accessories, Toys & Games, Animals & Pet Supplies, Product Add-Ons, Luggage & Bags, Cameras & Optics, Uncategorized, Office Supplies, Software, Furniture\n\nLightweight and durable 29-inch hardside roller suitcase featuring smooth spinner wheels, a secure travel lock, spacious internal organization, and scratch-resistant PC shell, ideal for checked luggage on holidays, trips, or business travel.\n\nThe category is ', 'completion': 'Luggage & Bags'}


In [19]:
def model_predict(item):
    inputs = tokenizer(item["prompt"],return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = base_model.generate(**inputs, max_new_tokens=8)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)

In [21]:
model_predict(test[0])

'\xa0Media, Arts & Entertainment.<|end_of_text|>'

# Configuration for Training

In [ ]:
# LoRA Parameters

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

In [ ]:
# Training parameters

train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS
)






In [ ]:
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_parameters,
    args=train_parameters
)

In [ ]:
# Fine-tune!
fine_tuning.train()

# Push our fine-tuned model to Hugging Face
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to the hub: {PROJECT_RUN_NAME}")

In [ ]:
if LOG_TO_WANDB:
  wandb.finish()